In [0]:
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_transactions_raw q;


In [0]:
SELECT * FROm demo_banking_silver.qdp_locations_all.hub_address h
  WHERE h.tennant_id =  100;

In [0]:
DELETE FROM demo_banking_silver.qdp_locations_all.sat_address s WHERE s.record_source_id = 100;


In [0]:
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw q;

In [0]:
SELECT
--  sat_address_id,
  h.address_id AS address_id,
--  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.postal_code AS postal_code,
  q.data_source_code AS data_source_code,
--  q.data_source_concept_id AS data_source_concept_id,
--  q.data_source_raw_code AS data_source_raw_code,
--  q.data_source_raw_concept_id AS data_source_raw_concept_id,
  q.type_code AS type_code,
--  q.type_concept_id AS type_concept_id,
--  q.type_raw_code AS type_raw_code,
--  q.type_raw_concept_id AS type_raw_concept_id,
  q.flatnumber_housenumber_housename AS flatnumber_housenumber_housename,
  q.house_number AS house_number,
  q.house_name AS house_name,
  q.building_number AS building_number,
  q.building_name AS building_name,
  q.address_line1 AS address_line1,
  q.address_line2 AS address_line2,
  q.address_line3 AS address_line3,
  q.address_line4 AS address_line4,
  q.address_line5 AS address_line5,
--  q.address_line6 AS address_line6,
--  q.address_line7 AS address_line7,
--  q.address_line8 AS address_line8,
  q.street AS street,
  q.district AS district,
  q.city AS city,
  q.county AS county,
  q.state AS state,
  q.country AS country,
  q.country_code AS country_code,
--  q.country_concept_id AS country_concept_id,
--  q.country_code_raw AS country_code_raw,
--  q.country_raw_concept_id AS country_raw_concept_id,
  q.full_address AS full_address,
--  q.uprn AS uprn,
--  q.what3words AS what3words,
  try_cast(q.latitude AS DOUBLE) AS latitude,
  try_cast(q.longitude AS DOUBLE) AS longitude,
--  q.directions AS directions,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw q
JOIN demo_banking_silver.qdp_locations_all.hub_address h
  ON h.address_entity_id = q.address_entity_id;


In [0]:
%sql
-- 01. Create variables for use in this notebook


-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;



DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;




DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;


-- NEED TO CREATE DATA PRODUCT SCHEMAS

--02. Back out data from Tennant before populating with new data

--02-0. Remove all LINK records for this Tennant
DELETE FROM demo_banking_silver.qdp_links_all.link_individual_address_history WHERE tennant_id = 100;
DELETE FROM demo_banking_silver.qdp_links_all.link_organisation_address_history WHERE tennant_id = 100;

--02-1. Remove all Location Address records for this Tennant
DELETE FROM demo_banking_silver.qdp_locations_all.sat_address s
  WHERE s.record_source_id =  100;

DELETE FROM demo_banking_silver.qdp_locations_all.hub_address h
  WHERE h.tennant_id =  100;

--02-2. Remove all Transaction records for this Tennant
DELETE FROM demo_banking_silver.qdp_transactions_all.sat_transaction s
 WHERE s.transaction_id IN (
  SELECT h.transaction_id 
    FROM demo_banking_silver.qdp_transactions_all.hub_transaction h 
    WHERE h.tennant_id = 100)
;




DELETE FROM demo_banking_silver.qdp_transactions_all.hub_transaction WHERE tennant_id = 100;


--02-3. Remove all Account records for this Tennant
DELETE FROM demo_banking_silver.qdp_accounts_all.sat_account s
  WHERE s.record_source_id = 100
-- WHERE s.account_id IN (
--  SELECT h.account_id 
--    FROM demo_banking_silver.qdp_accounts.hub_account h 
--    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 100;


--02-4. Remove all Individual Customer records for this Tennant
DELETE FROM demo_banking_silver.qdp_individual_customers.sat_individual_customers s
 WHERE s.individual_customer_id IN (
  SELECT h.individual_customer_id
    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers WHERE tennant_id = 100;



--02-5. Remove all Individual records for this Tennant
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_human_name s
 WHERE s.record_source_id = 100
;

DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual s
 WHERE s.individual_id IN (
  SELECT h.individual_id 
    FROM demo_banking_silver.qdp_individuals_all.hub_individual h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_individuals_all.hub_individual WHERE tennant_id = 100;



--02-6. Remove all Organisation Customer records for this Tennant
DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_risk_ratings s
  WHERE s.record_source_id =  100
;

DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_credit_ratings s
  WHERE s.record_source_id =  100
;

DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customers s
 WHERE s.organisation_customer_id IN (
  SELECT h.organisation_customer_id
    FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers WHERE tennant_id = 100;


--02-7. Remove all Organisation records for this Tennant
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_risk_ratings s
  WHERE s.record_source_id =  100
;

DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation_credit_ratings s
  WHERE s.record_source_id =  100
;


DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation s
  WHERE s.record_source_id =  100
-- WHERE s.organisation_id IN (
--  SELECT h.organisation_id 
--    FROM demo_banking_silver.qdp_organisations_all.hub_organisation h 
--    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_organisations_all.hub_organisation WHERE tennant_id = 100;



-- Populate the location_all Data Product
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw;



-- 03-1. Insert into hub_address
INSERT INTO demo_banking_silver.qdp_locations_all.hub_address
(address_entity_id, tennant_id, period_start, period_end)
SELECT address_entity_id, 100, period_start, period_end
FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw;

SELECT * FROM demo_banking_silver.qdp_locations_all.hub_address WHERE tennant_id = 100;


-- 03-2. Insert into sat_address
INSERT INTO demo_banking_silver.qdp_locations_all.sat_address
    (sat_address_id, 
     address_id, 
     load_datetime, 
     record_source_id, 
     postal_code,
     data_source_code,
--     data_source_concept_id,
--     data_source_raw_code,
--     data_source_raw_concept_id,
     type_code,
--     type_concept_id,
--     type_raw_code,
--     type_raw_concept_id,
     flatnumber_housenumber_housename,
     house_number,
     house_name,
     building_number,
     building_name,
     address_line1,
     address_line2,
     address_line3,
     address_line4,
     address_line5,
--     address_line6,
--     address_line7,
--     address_line8,
     street,
     district,
     city,
     county,
     state,
     country,
     country_code,
--     country_concept_id,
--     country_raw_code,
--     country_raw_concept_id,
     full_address,
--     uprn,
--     what3words,
     latitude,
     longitude,
--     directions,
     period_start,
     period_end
    )
SELECT
  monotonically_increasing_id() AS sat_address_id,
  h.address_id AS address_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.postal_code AS postal_code,
  q.data_source_code AS data_source_code,
--  q.data_source_concept_id AS data_source_concept_id,
--  q.data_source_raw_code AS data_source_raw_code,
--  q.data_source_raw_concept_id AS data_source_raw_concept_id,
  q.type_code AS type_code,
--  q.type_concept_id AS type_concept_id,
--  q.type_raw_code AS type_raw_code,
--  q.type_raw_concept_id AS type_raw_concept_id,
  q.flatnumber_housenumber_housename AS flatnumber_housenumber_housename,
  q.house_number AS house_number,
  q.house_name AS house_name,
  q.building_number AS building_number,
  q.building_name AS building_name,
  q.address_line1 AS address_line1,
  q.address_line2 AS address_line2,
  q.address_line3 AS address_line3,
  q.address_line4 AS address_line4,
  q.address_line5 AS address_line5,
--  q.address_line6 AS address_line6,
--  q.address_line7 AS address_line7,
--  q.address_line8 AS address_line8,
  q.street AS street,
  q.district AS district,
  q.city AS city,
  q.county AS county,
  q.state AS state,
  q.country AS country,
  q.country_code AS country_code,
--  q.country_concept_id AS country_concept_id,
--  q.country_code_raw AS country_code_raw,
--  q.country_raw_concept_id AS country_raw_concept_id,
  q.full_address AS full_address,
--  q.uprn AS uprn,
--  q.what3words AS what3words,
  try_cast(q.latitude AS DOUBLE) AS latitude,
  try_cast(q.longitude AS DOUBLE) AS longitude,
--  q.directions AS directions,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw q
JOIN demo_banking_silver.qdp_locations_all.hub_address h
  ON h.address_entity_id = q.address_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_locations_all.sat_address WHERE record_source_id = 100;

--  04-0. Create individuals_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw;


-- 04-1. Create the  hub_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.hub_individual
(individual_entity_id, tennant_id, national_id, period_start, period_end)
SELECT individual_entity_id, 100, national_id, period_start, period_end
FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw  ind
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.hub_individual WHERE tennant_id = 100;

-- 04-2. Create the  sat_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_individual
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
--     data_source_code,
--     data_source_concept_id,
--     data_source_raw_code,
--     data_source_raw_concept_id,
     birth_date,
--     birth_datetime,
--     birth_date_numeric,
--     estimated_birth_date,
--     estimated_birth_date_numeric,
--     is_multiple_birth,
--     multiple_birth_number,
     birth_place_city,
     birth_place_district,
     birth_place_country,
     birth_place_country_code,
--     photo,
--     is_deceased,
--     deceased_datetime,
--     deceased_date_numeric,
     gender_code,
--     gender_concept_id,
--     gender_raw_code,
--     gender_raw_concept_id,
     marital_status_code,
--     marital_status_concept_id,
--     marital_status_raw_code,
--     marital_status_raw_concept_id,
     ethnicity_code,
--     ethnicity_concept_id,
--     ethnicity_raw_code,
--     ethnicity_raw_concept_id,
--     religion_code,
--     religion_concept_id,
--     religion_raw_code,
--     religion_raw_concept_id,
     occupation_code,
--     occupation_concept_id,
--     occupation_raw_code,
--     occupation_raw_concept_id,
     nationality_code,
--     nationality_concept_id,
--     nationality_raw_code,
--     nationality_raw_concept_id,
     country_of_residence_code,
--     country_of_residence_concept_id,
--     country_of_residence_raw_code,
--     country_of_residence_raw_concept_id,
     primary_address_id,
     annual_income,
     number_of_dependants,
--     graduation_date,
     is_employee,
--     is_ex_employee,
     is_graduate,
--     is_interpreter_required,
--     is_employed,
--     is_retired,
--     is_student,
--     is_individual_merged,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
--  q.data_source_code,
--  q.data_source_concept_id,
--  q.data_source_raw_code,
--  q.data_source_raw_concept_id,
  q.birth_date AS birth_date,
  --q.birth_datetime AS birth_datetime,
  --q.birth_date_numeric AS birth_date_numeric,
  --q.estimated_birth_date AS estimated_birth_date,
  --q.estimated_birth_date_numeric AS estimated_birth_date_numeric,
  --q.is_multiple_birth AS is_multiple_birth,
  --q.multiple_birth_number AS multiple_birth_number,
  q.birth_place_city AS birth_place_city,
  q.birth_place_district AS birth_place_district,
  q.birth_place_country AS birth_place_country,
  q.birth_place_country_code AS birth_place_country_code,
 -- q.photo AS photo,
 -- q.is_deceased AS is_deceased,
 -- q.deceased_datetime AS deceased_datetime,
 -- q.deceased_date_numeric AS deceased_date_numeric,
  q.gender_code AS gender_code,
--  q.gender_concept_id AS gender_concept_id,
--  q.gender_raw_code AS gender_raw_code,
--  q.gender_raw_concept_id AS gender_raw_concept_id,
  q.marital_status_code AS marital_status_code,
--  q.marital_status_concept_id AS marital_status_concept_id,
--  q.marital_status_raw_code AS marital_status_raw_code,
--  q.marital_status_raw_concept_id AS marital_status_raw_concept_id,
  q.ethnicity_code AS ethnicity_code,
--  q.ethnicity_concept_id AS ethnicity_concept_id,
--  q.ethnicity_raw_code AS ethnicity_raw_code,
--  q.ethnicity_raw_concept_id AS ethnicity_raw_concept_id,
--  q.religion_code AS religion_code,
--  q.religion_concept_id AS religion_concept_id,
--  q.religion_raw_code AS religion_raw_code,
--  q.religion_raw_concept_id AS religion_raw_concept_id,
  q.occupation_code AS occupation_code,
--  q.occupation_concept_id AS occupation_concept_id,
--  q.occupation_raw_code AS occupation_raw_code,
--  q.occupation_raw_concept_id AS occupation_raw_concept_id,
  q.nationality_code AS nationality_code,
--  q.nationality_concept_id AS nationality_concept_id,
--  q.nationality_raw_code AS nationality_raw_code,
--  q.nationality_raw_concept_id AS nationality_raw_concept_id,
  q.country_of_residence_code AS country_of_residence_code,
--  q.country_of_residence_concept_id AS country_of_residence_concept_id,
--  q.country_of_residence_raw_code AS country_of_residence_raw_code,
--  q.country_of_residence_raw_concept_id AS country_of_residence_raw_concept_id,
  q.primary_address_id AS primary_address_id,
  q.annual_income AS annual_income,
  q.number_of_dependants AS number_of_dependants,
--  q.graduation_date AS graduation_date,
  q.is_employee AS is_employee,
--  q.is_ex_employee AS is_ex_employee,
  q.is_graduate AS is_graduate,
--  q.is_interpreter_required AS is_interpreter_required,
--  q.is_employed AS is_employed,
--  q.is_retired AS is_retired,
--  q.is_student AS is_student,
--  q.is_individual_merged AS is_individual_merged,
  try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_individual WHERE record_source_id = 100;


-- 04-4-1. Create the table of Individual Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.name_given1 AS given1,
    q.name_given2 AS given2,
    q.name_family AS family,
    '' AS full_given,
    q.name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;

-- 04-4-1. Create the table of Individual Alias1 Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.alias1_name_given1 AS given1,
    q.alias1_name_given2 AS given2,
    q.alias1_name_family AS family,
    '' AS full_given,
    q.alias1_name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id
  WHERE q.alias1_name_full IS NOT NULL;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;




-- 04-4-2. Create the table of Individual Alias2 Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.alias2_name_given1 AS given1,
    q.alias2_name_given2 AS given2,
    q.alias2_name_family AS family,
    '' AS full_given,
    q.alias2_name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id
  WHERE q.alias2_name_full IS NOT NULL;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;



-- 04-4-3.  Create the table of associated Addresses for an Individual

INSERT INTO demo_banking_silver.qdp_links_all.link_individual_address_history (
    tennant_id,
    individual_id,
    address_id,
    is_primary_address,
    individual_entity_id,
    address_entity_id,
    use_code,
    period_start,
    period_end)
  SELECT
    try_cast(100 AS BIGINT) AS tennant_id,
     hi.individual_id AS individual_id,
     ha.address_id AS address_id,
    true AS is_primary_address,
     hi.individual_entity_id AS individual_entity_id,
     ha.address_entity_id AS address_entity_id,
    'Home Address' AS use_code,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end
 FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual hi
    ON hi.individual_entity_id = q.individual_entity_id
  JOIN demo_banking_silver.qdp_locations_all.hub_address ha
    ON ha.address_entity_id = q.primary_address_entity_id
;

SELECT * FROM demo_banking_silver.qdp_links_all.link_individual_address_history WHERE tennant_id = 100;


-- 05-0. Create organisations_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw;


-- 05-1. Create the  hub_organisation table records
INSERT INTO demo_banking_silver.qdp_organisations_all.hub_organisation
(organisation_entity_id, 
 tennant_id, 
 period_start, 
 period_end)
SELECT q.organisation_entity_id, 
       100,
       q.period_start, 
       q.period_end
  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
;

SELECT * FROM demo_banking_silver.qdp_organisations_all.hub_organisation WHERE tennant_id = 100;


-- 05-2. Create the  sat_organisation table records

INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     data_source_code,
     type_code,
     status_code,
     organisation_name,
     trading_name,
     duns_number,
     organisation_description,
     organisation_level,
     parent_organisation_id,
     primary_adddress_id,
     company_category_code,
     company_registration_name,
     company_registration_number,
     company_registration_date,
     company_registration_country_code,
     primary_operation_country_code,
     vat_number,
     risk_rating_code,
     risk_rating,
     risk_rating_number,
     credit_rating_code,
     credit_rating,
     credit_rating_number,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.data_source_code AS data_source_code,
  q.type_code AS type_code,
  q.status_code AS status_code,
  q.organisation_name AS organisation_name,
  q.trading_name AS trading_name,
  q.duns_number AS duns_number,
  q.organisation_description AS organisation_description,
  try_cast(q.organisation_level AS INT) AS organisation_level,
  try_cast(0 AS BIGINT) AS parent_organisation_id,
  try_cast(q.primary_adddress_id as bigint) AS primary_adddress_id,
  q.company_category_code AS company_category_code,
  q.company_registration_name AS company_registration_name,
  q.company_registration_number AS company_registration_number,
  try_to_timestamp(q.company_registration_date, 'dd-MM-yyyy') AS company_registration_date,
  q.company_registration_country_code AS company_registration_country_code,
  q.primary_operation_country_code AS primary_oprtation_country_code,
  q.vat_number AS vat_number,
  'Risk Rating' AS risk_rating_code,
  q.risk_rating AS risk_rating,
  try_cast(q.risk_rating AS INT) AS risk_rating_number,
  'Internal Credit Rating' AS credit_rating_code,
  q.internal_credit_rating AS credit_rating,
  try_cast(q.internal_credit_rating AS INT) AS credit_rating_number,
  try_to_timestamp(q.period_start, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(q.period_end, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;

SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation WHERE record_source_id = 100;

-- 05-2-1. Create the Risk Rating entry for the Organisation in the sat_organisation_risk_ratings table
INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation_risk_ratings (
    organisation_id,
    load_datetime,
    record_source_id,
    risk_rating_code,
    risk_rating,
    risk_rating_number,
    period_start,
    period_end)
  SELECT
    h.organisation_id AS organisation_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    'Risk Rating' AS risk_rating_code,
    q.risk_rating AS risk_rating,
    try_cast(q.risk_rating AS INT) AS risk_rating_number,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
  JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
    ON h.organisation_entity_id = q.organisation_entity_id
  WHERE q.risk_rating IS NOT NULL
  
;

SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation_risk_ratings WHERE record_source_id = 100;




-- 05-2-2. Create the Internal Credit Rating entry for the Organisation in the sat_organisation_risk_ratings table
INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation_credit_ratings (
    organisation_id,
    load_datetime,
    record_source_id,
    credit_rating_code,
    credit_rating,
    credit_rating_number,
    period_start,
    period_end)
  SELECT
    h.organisation_id AS organisation_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    'Internal Credit Rating' AS credit_rating_code,
    q.internal_credit_rating AS credit_rating,
    try_cast(q.internal_credit_rating AS INT) AS credit_rating_number,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
  JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
    ON h.organisation_entity_id = q.organisation_entity_id
  WHERE q.risk_rating IS NOT NULL
  
;

SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation_credit_ratings WHERE record_source_id = 100;



-- 06-0. Create organisation_customers Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
    WHERE q.is_customer = true;


-- 06-1. Create the hub_organisation_customers table records
INSERT INTO demo_banking_silver.qdp_organisation_customers.hub_organisation_customers
(organisation_customer_entity_id, 
 tennant_id, 
 --address_entity_id, 
 organisation_entity_id, 
 organisation_id,
 period_start, 
 period_end)
SELECT q.organisation_entity_id, 
       100, 
--       q.address_entity_id, 
       q.organisation_entity_id, 
       h.organisation_id,
       q.period_start, 
       q.period_end
  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
    JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
      ON h.organisation_entity_id = q.organisation_entity_id
  WHERE q.is_customer = TRUE
;

SELECT * FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers 
  WHERE tennant_id = 100;

-- 06-2. Create the sat_organisation_customers table records
INSERT INTO demo_banking_silver.qdp_organisation_customers.sat_organisation_customers 
    ( 
     organisation_customer_id, 
     load_datetime, 
     record_source_id,
     risk_rating_code,
     risk_rating,
     risk_rating_number,
     credit_rating_code,
     credit_rating,
     credit_rating_number,
     period_start,
     period_end 
    )
SELECT
  hcust.organisation_customer_id AS organisation_customer_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  'Risk Rating' AS risk_rating_code,
  q.risk_rating AS risk_rating,
  try_cast(q.risk_rating AS INT) AS risk_rating_number,
  'Internal Credit Rating' AS credit_rating_code,
  q.internal_credit_rating AS credit_rating,
  try_cast(q.internal_credit_rating AS INT) AS credit_rating_number,
  try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
JOIN demo_banking_silver.qdp_organisation_customers.hub_organisation_customers hcust
  ON hcust.organisation_customer_entity_id = q.organisation_entity_id
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;



SELECT * FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customers 
  WHERE record_source_id = 100;


-- 06-2-1. Create the Risk Rating entry for the Organisation in the sat_organisation_risk_ratings table
INSERT INTO demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_risk_ratings (
    organisation_customer_id,
    load_datetime,
    record_source_id,
    risk_rating_code,
    risk_rating,
    risk_rating_number,
    period_start,
    period_end)
  SELECT
    hcust.organisation_customer_id AS organisation_customer_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    'Risk Rating' AS risk_rating_code,
    q.risk_rating AS risk_rating,
    try_cast(q.risk_rating AS INT) AS risk_rating_number,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
  JOIN demo_banking_silver.qdp_organisation_customers.hub_organisation_customers hcust
    ON hcust.organisation_entity_id = q.organisation_entity_id
  WHERE q.risk_rating IS NOT NULL
  
;


SELECT * FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_risk_ratings WHERE record_source_id = 100;



-- 06-2-2. Create the Credit Rating entry for the Organisation in the sat_organisation_credit_ratings table
INSERT INTO demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_credit_ratings (
    organisation_customer_id,
    load_datetime,
    record_source_id,
    credit_rating_code,
    credit_rating,
    credit_rating_number,
    period_start,
    period_end)
  SELECT
    hcust.organisation_customer_id AS organisation_customer_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    'Internal Credit Rating' AS credit_rating_code,
    q.internal_credit_rating AS credit_rating,
    try_cast(q.internal_credit_rating AS INT) AS credit_rating_number,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
  JOIN demo_banking_silver.qdp_organisation_customers.hub_organisation_customers hcust
    ON hcust.organisation_entity_id = q.organisation_entity_id
  WHERE q.risk_rating IS NOT NULL
  
;


SELECT * FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customer_credit_ratings WHERE record_source_id = 100;


-- 07-0. Create accounts_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_raw q
;

-- 07-1. Create the hub_account table records
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account
(account_entity_id, 
 tennant_id, 
 organisation_entity_id, 
 period_start, 
 period_end)
SELECT q.account_entity_id, 
       100, 
       q.organisation_entity_id,
       q.period_start, 
       q.period_end
  FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_raw q
  JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
    ON h.organisation_entity_id = q.organisation_entity_id
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account 
  WHERE tennant_id = 100;


-- 07-2. Create the sat_organisation_customers table records
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account 
    ( 
     account_id, 
     load_datetime, 
     record_source_id,
--     sort_code,
     account_number,
     account_name,
--     iban,
--     swiftbic,
     data_source_code,
     type_code,
     balance,
     overdraft_limit,
     currency_code,
--     branch_code,
     opened_datetime,
--     closed_datetime,
     status_code,
     country_code,
     loan_original_amount,
     loan_amount,
     colattoral_last_valued_date,
     loan_location,
     cre_rating,
     gaurantor_amount,
     guarantor_name,
     gaurantor_rating,
     is_account_alarm,
     is_bank_account,
     is_business_account,
     is_customer_account,
     is_counterparty_account,
     is_frozen,
     is_closed,
     is_open,
     is_excluded_from_monitoring,
     period_start,
     period_end 
    )
SELECT
  hacc.account_id AS account_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
--  sort_code,
  q.account_number,
  q.account_name,
--  iban,
--  swiftbic,
  q.data_source_code,
  q.type_code,
  q.balance,
  q.overdraft_limit,
  q.currency_code,
--  branch_code,
  q.opened_datetime,
--  closed_datetime,
  q.status_code,
  q.country_code,
  q.loan_value AS loan_original_amount,
  q.loan_value AS loan_amount,
  q.colattoral_last_valued_date AS colattoral_last_valued_date,
  q.loan_location as loan_location,
  q.cre_rating AS cre_rating,
  q.guarantoor_amount AS gaurantor_amount,
  q.guarantor_name AS guarantor_name,
  q.guarantor_rating AS gaurantor_rating,
  q.is_account_alarm,
  q.is_bank_account,
  q.is_business_account,
  q.is_customer_acccount,
  false,
  false,
  false,
  true,
  false,
  try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_raw q
JOIN demo_banking_silver.qdp_accounts_all.hub_account hacc
  ON hacc.account_entity_id = q.account_entity_id
;


SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account 
  WHERE record_source_id = 100;



-- 07-0. Create accounts_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_transactions_raw q
;


-- 8.1 Insert into hub_transactions
INSERT INTO demo_banking_silver.qdp_transactions_all.hub_transaction
(transaction_entity_id, 
 tennant_id,
 bene_account_entity_id,
 orig_account_entity_id,
 bene_account_id,
 orig_account_id,
 transaction_id_raw)
 SELECT q.transaction_entity_id, 
       100,
       q.bene_account_entity_id,
       q.orig_account_entity_id,
       haccbene.account_id,
       haccorig.account_id,
       q.transaction_entity_id
FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_transactions_raw q
 JOIN demo_banking_silver.qdp_accounts_all.hub_account haccbene
   ON haccbene.account_entity_id = q.bene_account_entity_id
 JOIN demo_banking_silver.qdp_accounts_all.hub_account haccorig
   ON haccorig.account_entity_id = q.orig_account_entity_id
;

SELECT * FROM demo_banking_silver.qdp_transactions_all.hub_transaction WHERE tennant_id = 100;

-- 16.2 Insert into sat_transactions
INSERT INTO demo_banking_silver.qdp_transactions_all.sat_transaction
    (transaction_id, 
     load_datetime, 
     record_source_id,
     account_number,
     payment_method_raw_code,
     data_source_raw_code,
     debit_or_credit_raw_code,
     amount,
     datetime,
     original_country_raw_code,
     country_code,
     original_account_number,
     original_amount,
     original_currency_raw_code,
     currency_code
    )
SELECT
  h.transaction_id AS transaction_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.Bene_Account_No,
  q.Transaction_Type AS type_code,
  q.Source_System,
  q.Transaction_Type,
  q.Bene_Amount,
  TRY_TO_DATE(q.Date, "yyyy-MM-dd") AS transaction_date_time,
  q.Orig_Country,
  q.Bene_Country,
  q.Orig_Account_No,
  q.Orig_Amount,
  q.Orig_Currency,
  q.Base_currency

FROM demo_banking_bronze.risk_bcbs239_demo.risk_account_transactions_raw q
JOIN demo_banking_silver.qdp_transactions_all.hub_transaction h
  ON h.transaction_entity_id = q.transaction_entity_id
;  


SELECT * FROM demo_banking_silver.qdp_transactions_all.sat_transaction where record_source_id = 100;

/**





**/


In [0]:
SELECT * FROM demo_banking_silver.qdp_transactions_all.view_transactions WHERE record_source_id = 100;